# Estudo Pandas

---

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de display para facilitar inspeção
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Dicionário de Dados

| Base | Coluna | Tipo Esperado | Descrição |
|------|--------|---------------|-----------|
| vendas | ID_Pedido | int | Identificador único do pedido |
| vendas | Data | datetime | Data da transação |
| vendas | Loja | str | Cidade da loja (pode conter inconsistências de case) |
| vendas | Produto | str | Nome do produto comercializado |
| vendas | Preco_Unitario | float | Valor unitário em R$ |
| vendas | Qtd | int | Quantidade vendida |
| vendas | Cliente | str | Código anonimizado do cliente |
| vendas | Data_Base | datetime | Data de snapshot da base (quase 100% nulo) |
| gerentes | Loja | str | Cidade da loja (chave para join) |
| gerentes | Gerente | str | Nome do responsável |
| gerentes | Meta_Mensal | int | Meta de faturamento em R$ |

## 3. Carregando dados e definindo variáveis

**Decisões:**
- `low_memory=False` força inferência de tipos em uma única passada (base > 100k linhas).
- `parse_dates` já converte colunas de data no carregamento, evitando conversão posterior.

In [2]:
df_vendas = pd.read_csv(
    'vendas_tech.csv',
    sep=',',
    low_memory=False,
    parse_dates=['Data', 'Data_Base']  # conversão imediata de datas
)

df_gerentes = pd.read_excel('gerentes_lojas.xlsx')

# Sanity check: schema e amostra
print(f'Vendas: {df_vendas.shape[0]} rows, {df_vendas.shape[1]} cols')
print(f'Gerentes: {df_gerentes.shape[0]} rows, {df_gerentes.shape[1]} cols')
#------------------------------------------------------------------
print("=== VENDAS ===")
display(df_vendas)
print("\n")
print("=== GERENTES ===")
display(df_gerentes)
print(df_gerentes)


Vendas: 100100 rows, 8 cols
Gerentes: 7 rows, 3 cols
=== VENDAS ===


,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Data_Base
0,1,2023-06-08,São Paulo,Mouse Gamer,120.0,1,Cliente_4095,2025-01-01
1,2,2023-03-01,Belo Horizonte,iPhone 14,5500.0,1,Cliente_8750,NaT
2,3,2023-02-25,NaN,"Monitor 27""",1200.0,1,Cliente_14859,NaT
3,4,2024-11-19,RIO DE JANEIRO,Mouse Gamer,120.0,2,Cliente_17343,NaT
4,5,2024-01-27,Rio de Janeiro,Smartphone Samsung,2200.0,1,Cliente_23377,NaT
...,...,...,...,...,...,...,...,...
100095,94091,2023-10-20,Porto Alegre,iPhone 14,5500.0,1,Cliente_11755,NaT
100096,52883,2024-03-17,Porto Alegre,Notebook Dell,3500.0,2,Cliente_12879,NaT
100097,65070,2023-06-19,Belo Horizonte,Smartphone Samsung,2200.0,2,Cliente_8160,NaT
100098,94031,2024-06-14,Salvador,iPhone 14,5500.0,2,Cliente_28545,NaT




=== GERENTES ===


,Loja,Gerente,Meta_Mensal
0,São Paulo,Carlos,50000
1,Rio de Janeiro,Fernanda,60000
2,Curitiba,Roberto,45000
3,Belo Horizonte,Juliana,55000
4,Recife,Marcos,48000
5,Porto Alegre,Pedro,42000
6,Salvador,Ana,52000


             Loja   Gerente  Meta_Mensal
0       São Paulo    Carlos        50000
1  Rio de Janeiro  Fernanda        60000
2        Curitiba   Roberto        45000
3  Belo Horizonte   Juliana        55000
4          Recife    Marcos        48000
5    Porto Alegre     Pedro        42000
6        Salvador       Ana        52000


## 4. Inspeção Inicial (EDA Rápida)

Aqui não se explica o que `head()` faz. A célula é autoexplicativa pelo título da seção.

In [3]:
df_vendas.head()

,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Data_Base
0,1,2023-06-08,São Paulo,Mouse Gamer,120.0,1,Cliente_4095,2025-01-01
1,2,2023-03-01,Belo Horizonte,iPhone 14,5500.0,1,Cliente_8750,NaT
2,3,2023-02-25,NaN,"Monitor 27""",1200.0,1,Cliente_14859,NaT
3,4,2024-11-19,RIO DE JANEIRO,Mouse Gamer,120.0,2,Cliente_17343,NaT
4,5,2024-01-27,Rio de Janeiro,Smartphone Samsung,2200.0,1,Cliente_23377,NaT


In [4]:
df_vendas.tail()

,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Data_Base
100095,94091,2023-10-20,Porto Alegre,iPhone 14,5500.0,1,Cliente_11755,NaT
100096,52883,2024-03-17,Porto Alegre,Notebook Dell,3500.0,2,Cliente_12879,NaT
100097,65070,2023-06-19,Belo Horizonte,Smartphone Samsung,2200.0,2,Cliente_8160,NaT
100098,94031,2024-06-14,Salvador,iPhone 14,5500.0,2,Cliente_28545,NaT
100099,36755,2023-01-03,São Paulo,Notebook HP,3200.0,2,Cliente_3274,NaT


In [5]:
display(df_gerentes)

,Loja,Gerente,Meta_Mensal
0,São Paulo,Carlos,50000
1,Rio de Janeiro,Fernanda,60000
2,Curitiba,Roberto,45000
3,Belo Horizonte,Juliana,55000
4,Recife,Marcos,48000
5,Porto Alegre,Pedro,42000
6,Salvador,Ana,52000


### 4.1 Schema e Qualidade

`info()` + `describe()` para detectar nulos, tipos incorretos e outliers de primeira ordem.

In [6]:
df_vendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 100100 entries, 0 to 100099
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   ID_Pedido       100100 non-null  int64         
 1   Data            100100 non-null  datetime64[us]
 2   Loja            98099 non-null   str           
 3   Produto         100100 non-null  str           
 4   Preco_Unitario  100100 non-null  float64       
 5   Qtd             100100 non-null  int64         
 6   Cliente         100100 non-null  str           
 7   Data_Base       1 non-null       datetime64[us]
dtypes: datetime64[us](2), float64(1), int64(2), str(3)
memory usage: 9.5 MB


In [7]:
df_vendas.describe()

,ID_Pedido,Data,Preco_Unitario,Qtd,Data_Base
count,100100.000000,100100,100100.000000,100100.000000,1
mean,50004.810180,2023-12-30 05:01:35.664335,2000.204595,1.499101,2025-01-01 00:00:00
min,1.000000,2023-01-01 00:00:00,40.000000,1.000000,2025-01-01 00:00:00
25%,25008.750000,2023-06-29 00:00:00,120.000000,1.000000,2025-01-01 00:00:00
50%,50004.500000,2023-12-29 00:00:00,1200.000000,1.000000,2025-01-01 00:00:00
75%,75002.250000,2024-06-30 00:00:00,3200.000000,1.000000,2025-01-01 00:00:00
max,100000.000000,2024-12-30 00:00:00,5500.000000,10.000000,2025-01-01 00:00:00
std,28866.872543,NaN,1841.050733,1.241605,NaN


## 5. ETL — Limpeza

**Pipeline:**
1. Padronizar case da coluna `Loja` (RIO DE JANEIRO → Rio de Janeiro).
2. Dropar `Data_Base` (1 non-null em 100k linhas = sem valor analítico).
3. Tratar nulos em `Loja` (2% missing).
df_vendas.info()

In [8]:
# Cópia explícita para não mutar o original durante experimentos
df = df_vendas.copy()

# 1. Normalização de texto
df['Loja'] = df['Loja'].str.title()

# 2. Drop de coluna com cardinalidade/missing inviável
df = df.drop(columns=['Data_Base'])

# 3. Tratamento de nulos: como Loja é dimensão analítica, dropamos linhas sem loja
# (alternativa: imputar 'Desconhecido', mas isso quebraria o join com gerentes)
df = df.dropna(subset=['Loja'])

# 4. Duplicatas (considerando todas as colunas como chave lógica)
duplicatas = df.duplicated().sum()
print(f'Duplicatas encontradas: {duplicatas}')
if duplicatas > 0:
    df = df.drop_duplicates()

print(f'Base final: {df.shape[0]} rows')

Duplicatas encontradas: 100
Base final: 97999 rows


## 6. Feature Engineering (básica)

Criar métricas derivadas para análise.


In [9]:
df['Valor_Total'] = df['Preco_Unitario'] * df['Qtd']
df['Ano_Mes'] = df['Data'].dt.to_period('M').astype(str)  # para agrupamentos temporais

## 7. Integração (Join)

Merge com cadastro de gerentes para enriquecer a base.

In [10]:
df_final = df.merge(
    df_gerentes,
    on='Loja',
    how='left',
    indicator=True  # flag para auditoria de match
)

### Auditoria: todas as lojas bateram?
print(df_final['_merge'].value_counts())

_merge
both          79696
left_only     18303
right_only        0
Name: count, dtype: int64


## 8. Checkpoint

Exportar base limpa para próximas etapas (análise ou modelagem).

In [11]:
df_final['Ano_Mes'] = df_final['Ano_Mes'].astype(str)
df_final.to_parquet('vendas_clean.parquet', index=False)
# parquet preserva tipos (datetime, category) melhor que csv